#1. Read the two Silver tables

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "adb_ecommerce_de_7405607562556202"

products_df = spark.table(
    f"{catalog}.ecommerce_silver.products"
)

category_df = spark.table(
    f"{catalog}.ecommerce_silver.category_translation"
)

#2. Join them and create the product dimension

In [0]:
window_spec = Window.orderBy(products_df["product_id"])

gold_dim_product = (
    products_df
    .join(
        category_df,
        products_df["product_category_name"] ==
        category_df["product_category_name"],
        "left"
    )
    .drop(category_df["product_category_name"])
    .dropDuplicates(["product_id"])
    .withColumn(
        "product_key",
        F.row_number().over(window_spec)
    )
    .select(
        "product_key",
        products_df["product_id"],
        products_df["product_category_name"],
        category_df["product_category_name_english"],
        products_df["product_name_length"],
        products_df["product_description_length"],
        products_df["product_photos_qty"],
        products_df["product_weight_g"],
        products_df["product_length_cm"],
        products_df["product_height_cm"],
        products_df["product_width_cm"]
    )
)

#3. Write to Gold

In [0]:
gold_path = "abfss://ecommerce@mayankecommerce017.dfs.core.windows.net/gold/dim_product"

gold_dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ecommerce_gold.dim_product
USING DELTA
LOCATION '{gold_path}'
""")

In [0]:
spark.table(
    f"{catalog}.ecommerce_gold.dim_product"
).show(5, truncate=False)